Installing opendatasets library to take taking datasets from kaggle

In [0]:
%pip install opendatasets

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pyspark
import opendatasets as od
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

#### "Telco Customer Dataset"

%md
Implementing "Telco Customer Churn Dataset" using API token. User data is specified in kaggle.json file in order to insert them automatically. 

I chose this dataset because it's based on the original Telco Customer Churn dataset (provided by IBM)
and it also contains records with missing values which makes it more realistic.


In [0]:
od.download("https://www.kaggle.com/datasets/jethwaaatmik/telco-customer-churn-dataset")

Skipping, found downloaded files in "./telco-customer-churn-dataset" (use force=True to force download)


In [0]:
# Initializing Spark session
spark = SparkSession.builder.getOrCreate()

In [0]:
current_dir = os.getcwd()
path = "telco-customer-churn-dataset/telco_data.csv"
absolute_path = os.path.join(current_dir, path)

df = spark.read.csv(absolute_path, header = True)

# I am creating spark dataframe with subset of columns in order to reduce the size of the dataset
# so the data will be shown in the correct format
df_slim = df.select("customerID", "gender", "tenure", "Contract", "MonthlyCharges", "Churn", "InternetService")


In [0]:
df_slim.show(n = 10)

+----------+------+------+--------------+--------------+-----+---------------+
|customerID|gender|tenure|      Contract|MonthlyCharges|Churn|InternetService|
+----------+------+------+--------------+--------------+-----+---------------+
|7590-VHVEG|Female|   1.0|Month-to-month|         29.85|   No|            DSL|
|5575-GNVDE|  Male|  34.0|      One year|         56.95|   No|            DSL|
|3668-QPYBK|  Male|   2.0|Month-to-month|         53.85|  Yes|            DSL|
|7795-CFOCW|  Male|  45.0|      One year|          42.3|   No|            DSL|
|9237-HQITU|Female|   2.0|Month-to-month|          70.7|  Yes|    Fiber optic|
|9305-CDSKC|Female|   8.0|Month-to-month|         99.65|  Yes|    Fiber optic|
|1452-KIOVK|  Male|  22.0|Month-to-month|          89.1|   No|    Fiber optic|
|6713-OKOMC|Female|  10.0|Month-to-month|         29.75|   No|            DSL|
|7892-POOKP|Female|  NULL|Month-to-month|          NULL|  Yes|    Fiber optic|
|6388-TABGU|  Male|  62.0|      One year|         56

In [0]:
df_slim.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- tenure: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- MonthlyCharges: string (nullable = true)
 |-- Churn: string (nullable = true)
 |-- InternetService: string (nullable = true)



In [0]:
row_count = df_slim.count()
print("Total number of rows: ", row_count)

Total number of rows:  7043


In [0]:
for c in df_slim.columns:
  print(f"{c}: ", df.filter(f.col(c).isNull()).count())


customerID:  0
gender:  750
tenure:  2500
Contract:  0
MonthlyCharges:  1500
Churn:  0
InternetService:  1000


#### Adding extra column

I will add extra column based on Contract Category.
For each category (e.g. Month-to-month, One year) I will calculate average monthly charges.
Then the average value will be added to each value of the newly created column accordingly


In [0]:
contract_averages = {}   # dictionary to store average monthly charges per category

all_contract_categories = df_slim.select(df_slim.Contract).distinct().collect()
for row in all_contract_categories:
    category = row[0]           # takes name of category    
    df_slim_category = df_slim.filter(df_slim.Contract == category)  # temporary df with only one category
    avg_df = df_slim_category.agg(f.avg(df_slim_category.MonthlyCharges))
    avg = avg_df.collect()[0][0]  
    contract_averages[category] = avg    


categories = list(contract_averages.keys())
expr = f.when(f.col("Contract") == categories[0], contract_averages[categories[0]])
for category in categories[1:]:
    expr = expr.when(f.col("Contract") == category, contract_averages[category])

df_slim_category = df_slim.withColumn("avg_MonthlyCharges", expr)   # adding a new column


#### Filling missing values 
- When it comes to the internet service I want to fill missing values with a value that occurs most often in internet service column, because there is the highest probability that the missing value may be like this
- When it comes to monthly charges column I want to fill missing values with average charges based on contract type (column that I added before), because monthly charges mostly depend on the length of the contract 
( monthly charges for month-to-month should be a little bit higher due to the fact that the customer can resign in every moment)


In [0]:

# filling missing MonthlyCharges
df_slim_category = df_slim_category.withColumn("MonthlyCharges",f.coalesce(df_slim_category.MonthlyCharges,
f.round(df_slim_category.avg_MonthlyCharges,2)))
df_slim_category.show(n=10)


+----------+------+------+--------------+--------------+-----+---------------+------------------+
|customerID|gender|tenure|      Contract|MonthlyCharges|Churn|InternetService|avg_MonthlyCharges|
+----------+------+------+--------------+--------------+-----+---------------+------------------+
|7590-VHVEG|Female|   1.0|Month-to-month|         29.85|   No|            DSL| 66.49481566820255|
|5575-GNVDE|  Male|  34.0|      One year|         56.95|   No|            DSL| 65.06052188552188|
|3668-QPYBK|  Male|   2.0|Month-to-month|         53.85|  Yes|            DSL| 66.49481566820255|
|7795-CFOCW|  Male|  45.0|      One year|          42.3|   No|            DSL| 65.06052188552188|
|9237-HQITU|Female|   2.0|Month-to-month|          70.7|  Yes|    Fiber optic| 66.49481566820255|
|9305-CDSKC|Female|   8.0|Month-to-month|         99.65|  Yes|    Fiber optic| 66.49481566820255|
|1452-KIOVK|  Male|  22.0|Month-to-month|          89.1|   No|    Fiber optic| 66.49481566820255|
|6713-OKOMC|Female| 

In [0]:
## filling missing InternetService
grouped_services = df_slim_category.groupby('InternetService').count()
most_popular_service = grouped_services.sort("count", ascending=False).take(1)[0][0]
df_slim_category = df_slim_category.na.fill({'InternetService': most_popular_service})
df_slim_category.show(n=10)

+----------+------+------+--------------+--------------+-----+---------------+------------------+
|customerID|gender|tenure|      Contract|MonthlyCharges|Churn|InternetService|avg_MonthlyCharges|
+----------+------+------+--------------+--------------+-----+---------------+------------------+
|7590-VHVEG|Female|   1.0|Month-to-month|         29.85|   No|            DSL| 66.49481566820255|
|5575-GNVDE|  Male|  34.0|      One year|         56.95|   No|            DSL| 65.06052188552188|
|3668-QPYBK|  Male|   2.0|Month-to-month|         53.85|  Yes|            DSL| 66.49481566820255|
|7795-CFOCW|  Male|  45.0|      One year|          42.3|   No|            DSL| 65.06052188552188|
|9237-HQITU|Female|   2.0|Month-to-month|          70.7|  Yes|    Fiber optic| 66.49481566820255|
|9305-CDSKC|Female|   8.0|Month-to-month|         99.65|  Yes|    Fiber optic| 66.49481566820255|
|1452-KIOVK|  Male|  22.0|Month-to-month|          89.1|   No|    Fiber optic| 66.49481566820255|
|6713-OKOMC|Female| 

#### Filtering: customers whose gender is female

In [0]:
df_slim_female = df_slim_category.filter(df_slim_category.gender == "Female")
df_slim_female.show(n=10)

+----------+------+------+--------------+--------------+-----+---------------+------------------+
|customerID|gender|tenure|      Contract|MonthlyCharges|Churn|InternetService|avg_MonthlyCharges|
+----------+------+------+--------------+--------------+-----+---------------+------------------+
|7590-VHVEG|Female|   1.0|Month-to-month|         29.85|   No|            DSL| 66.49481566820255|
|9237-HQITU|Female|   2.0|Month-to-month|          70.7|  Yes|    Fiber optic| 66.49481566820255|
|9305-CDSKC|Female|   8.0|Month-to-month|         99.65|  Yes|    Fiber optic| 66.49481566820255|
|6713-OKOMC|Female|  10.0|Month-to-month|         29.75|   No|            DSL| 66.49481566820255|
|7892-POOKP|Female|  NULL|Month-to-month|         66.49|  Yes|    Fiber optic| 66.49481566820255|
|3655-SNQYZ|Female|  NULL|      Two year|         60.98|   No|    Fiber optic| 60.97703113135917|
|8191-XWSZG|Female|  52.0|      One year|         20.65|   No|             No| 65.06052188552188|
|4190-MFLUW|Female| 

#### SQL analysis

In [0]:
df_slim_category.createOrReplaceTempView("people_data")

In [0]:
grouped_services = spark.sql(
"SELECT InternetService, COUNT(*) AS How_many FROM people_data GROUP BY InternetService"                       
)                    

grouped_services.show()

+---------------+--------+
|InternetService|How_many|
+---------------+--------+
|    Fiber optic|    3660|
|            DSL|    2082|
|             No|    1301|
+---------------+--------+



In [0]:
grouped_contracts = spark.sql(
"SELECT Contract, COUNT(*) AS How_many FROM people_data GROUP BY Contract"                       
)   

grouped_contracts.show()

+--------------+--------+
|      Contract|How_many|
+--------------+--------+
|      One year|    1473|
|      Two year|    1695|
|Month-to-month|    3875|
+--------------+--------+



In [0]:
grouped_services.write.format("delta").saveAsTable("people_data_table")

#### My thoughts
-  I notice that customers with fiber optic pay higher rates
- It's surprising that the most popular contract is month-to-month despite of the fact that it's the most expensive one (when it comes to average monthly charges).
- If I had more time, I would start looking for some complex relationships, e.g. look for correlations of various features.
- What stood out was the surprisingly high number of missing values, which raises questions about data collection quality.